In [2]:
text = """TRANSMISSION #8841

Source: PROBE-Σ7
Signal Quality: 98.4% ▓▓▓▓▓▓▓▓▓░

Timestamp:
[2457-08-19T04:11:07Z]

Detected Objects:
◉ Unknown Star
◈ Metallic Structure
☄ Debris Field

Warning:
Radiation levels exceeded threshold ⚠

Last Recorded Message:
"Trajectory stable. Fuel reserves at 0.03%.
If this packet is received, continue mission."

Connection lost ⋯""" # just some randome text
tokens = text.encode("utf-8")
tokens = list(tokens)
print(tokens)
print(f"text length: {len(text)}")
print(f"{len(tokens)} tokens")

[84, 82, 65, 78, 83, 77, 73, 83, 83, 73, 79, 78, 32, 35, 56, 56, 52, 49, 10, 10, 83, 111, 117, 114, 99, 101, 58, 32, 80, 82, 79, 66, 69, 45, 206, 163, 55, 10, 83, 105, 103, 110, 97, 108, 32, 81, 117, 97, 108, 105, 116, 121, 58, 32, 57, 56, 46, 52, 37, 32, 226, 150, 147, 226, 150, 147, 226, 150, 147, 226, 150, 147, 226, 150, 147, 226, 150, 147, 226, 150, 147, 226, 150, 147, 226, 150, 147, 226, 150, 145, 10, 10, 84, 105, 109, 101, 115, 116, 97, 109, 112, 58, 10, 91, 50, 52, 53, 55, 45, 48, 56, 45, 49, 57, 84, 48, 52, 58, 49, 49, 58, 48, 55, 90, 93, 10, 10, 68, 101, 116, 101, 99, 116, 101, 100, 32, 79, 98, 106, 101, 99, 116, 115, 58, 10, 226, 151, 137, 32, 85, 110, 107, 110, 111, 119, 110, 32, 83, 116, 97, 114, 10, 226, 151, 136, 32, 77, 101, 116, 97, 108, 108, 105, 99, 32, 83, 116, 114, 117, 99, 116, 117, 114, 101, 10, 226, 152, 132, 32, 68, 101, 98, 114, 105, 115, 32, 70, 105, 101, 108, 100, 10, 10, 87, 97, 114, 110, 105, 110, 103, 58, 10, 82, 97, 100, 105, 97, 116, 105, 111, 110, 32, 1

In [3]:
from collections import Counter

def get_stats(ids):
    counts = Counter(zip(ids, ids[1:]))
    return counts

def merge(ids, pair, idx):
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i + 1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids

vocab_size = 276
num_merges = vocab_size - 256
ids = list(tokens)

merges = {}
for i in range(num_merges):
    stats = get_stats(ids)
    pair = max(stats, key=stats.get)
    idx = 256 + i
    print(f"merging {pair} into new token {idx}")
    ids = merge(ids, pair, idx)
    merges[pair] = idx


merging (226, 150) into new token 256
merging (256, 147) into new token 257
merging (257, 257) into new token 258
merging (10, 10) into new token 259
merging (101, 99) into new token 260
merging (101, 115) into new token 261
merging (101, 100) into new token 262
merging (111, 110) into new token 263
merging (116, 97) into new token 264
merging (58, 10) into new token 265
merging (260, 116) into new token 266
merging (105, 115) into new token 267
merging (116, 32) into new token 268
merging (258, 258) into new token 269
merging (262, 32) into new token 270
merging (267, 32) into new token 271
merging (101, 108) into new token 272
merging (105, 263) into new token 273
merging (117, 114) into new token 274
merging (99, 101) into new token 275


In [4]:
print(len(tokens))
print(len(ids))
print(f"compression ratio: {len(tokens) / len(ids):.2f}")

387
299
compression ratio: 1.29


In [6]:
vocab = {idx : bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

def decode(ids):
    tokens = b"".join(vocab[idx] for idx in ids)
    return tokens.decode("utf-8")

print(decode(ids))

TRANSMISSION #8841

Source: PROBE-Σ7
Signal Quality: 98.4% ▓▓▓▓▓▓▓▓▓░

Timestamp:
[2457-08-19T04:11:07Z]

Detected Objects:
◉ Unknown Star
◈ Metallic Structure
☄ Debris Field

Radiation levels exceeded threshold ⚠

Last Recorded Message:
"Trajectory stable. Fuel reserves at 0.03%.
If this packet is received, continue mission."

Connection lost ⋯


In [8]:
def encode(text):
    tokens = list(text.encode("utf-8"))
    while (len(tokens) > 1):
        stats = get_stats(tokens)
        pair = min(stats, key = lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break
        idx = merges[pair]
        tokens = merge(tokens, pair, idx)
    return tokens

print(encode("hello world!"))
print(decode(encode("hello world!")))

[104, 272, 108, 111, 32, 119, 111, 114, 108, 100, 33]
hello world!


In [9]:
import regex as re
gpt2pat = re.compile(r"""'s|'t|'re|'ve|'m|'ll|'d| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+""")

print(re.findall(gpt2pat, "hey, if you dived so deep into my code, here's my number - +918806996886!"))

['hey', ',', ' if', ' you', ' dived', ' so', ' deep', ' into', ' my', ' code', ',', ' here', "'s", ' my', ' number', ' -', ' +', '918806996886', '!']


In [10]:
!pip install tiktoken
import tiktoken

# GPT-2 (does not merge spaces)
enc = tiktoken.get_encoding("gpt2")
print(enc.encode("    zwix OP!!!"))

# GPT-4 (merges spaces)
enc = tiktoken.get_encoding("cl100k_base")
print(enc.encode("    zwix OP!!!"))

[220, 220, 220, 1976, 86, 844, 13349, 10185]
[262, 25398, 953, 13435, 12340]
